### Initial notebook for testing the repo

In [ ]:
# Notebook to compare visualisation results for NIfTI vs OME-zarr using Neuroglancer & ngff-zarr
import subprocess

inp_files = ['/Users/Vasilis/CMC_data/Moe/MRI/reoriented_FA.nii.gz',
             '/Users/Vasilis/Downloads/CMC_results/Moe_cc_Ret/slide_deck_to_mri.nii.gz',
            #  '/Users/Vasilis/Downloads/CMC_results/Moe_cc_Ret/Retardance/lowres/Slice_111_EnR_downsample_10_hdr.nii.gz',
             ]
out_names = ['DTI_FA_in_native',
             'PSOCT_Ret_in_DTI',
            #  'PSOCT_Ret_slice_111_in_DTI',
             ]
out_folder = 'Moe_cc_Ret'

inp_zarr = []
for file, name in zip(inp_files, out_names):
    inp_zarr.append(out_folder + '/' + name + '.zarr')
    subprocess.run(["ngff-zarr",
                    "-i", file,
                    "-o", inp_zarr[-1],
                    "--ome-zarr-version", str(0.5)
                    ])

subprocess.Popen(["python",
                "../visualize_zarr_ng/visualize_zarr.py",
                *inp_zarr,
                "--port", str(8050),
                "--name", *out_names])



In [ ]:
# split slide deck into slices (so they are resampled)
import subprocess, os
from pathlib import Path
data_dir = Path('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409')
out_folder = Path('/Users/Vasilis/Downloads/CMC_results/Zarr_test/sliced')
slide_deck = data_dir / 'Ret_slide_deck.nii.gz'

os.makedirs(out_folder, exist_ok=True)
subprocess.run(["fslsplit",
                str(slide_deck),
                str(out_folder / 'slice'),
                "-x"])

### Notebook for converting all 2D slices to Zarr for later 3D visualisation

In [4]:
import subprocess
from pathlib import Path

# ori_dir = Path('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409/Orientation/lowres')
ori_dir = Path('/Users/Vasilis/Downloads/CMC_results/Zarr_test/sliced')
mri_ref = Path('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409/dti_FA_in_PSOCT.nii.gz')
# out_folder = Path('/Users/Vasilis/Downloads/CMC_results/Zarr_test')
out_folder = Path('/Users/Vasilis/Downloads/CMC_results/Zarr_test/sliced')

inp_zarr = []
# convert reference MRI image into Zarr
out_name = out_folder / mri_ref.name.replace('.nii.gz', '.zarr')
inp_zarr.append(out_name)
if not out_name.exists():
    subprocess.run(["ngff-zarr",
                    "-i", mri_ref,
                    "-o", out_name,
                    "--ome-zarr-version", str(0.5)
                    ])

# convert Orientation slices into Zarr
slices = ori_dir.glob('*.nii.gz')
for sl in slices:
    out_name = out_folder / sl.name.replace('.nii.gz', '.zarr')
    inp_zarr.append(out_name)
    if not out_name.exists():
        subprocess.run(["ngff-zarr",
                        "-i", sl,
                        "-o", out_name,
                        "--ome-zarr-version", str(0.5)
                        ])

╭─────────────────────────────── NGFF OME-Zarr ────────────────────────────────╮
│ ∙∙∙ Loading input...                                                         │
╭─────────────────────────────── NGFF OME-Zarr ────────────────────────────────╮
│ ∙●∙ Loading input...                                                         │
╭─────────────────────────────── NGFF OME-Zarr ────────────────────────────────╮
│ ∙∙∙ Loading input...                                                         │
╭─────────────────────────────── NGFF OME-Zarr ────────────────────────────────╮
│ ●∙∙ Loading input...                                                         │
╭─────────────────────────────── NGFF OME-Zarr ────────────────────────────────╮
│ ∙∙● Loading input...                                                         │
╭─────────────────────────────── NGFF OME-Zarr ────────────────────────────────╮
╭─────────────────────────────── NGFF OME-Zarr ────────────────────────────────╮
╭───────────────────────────

In [5]:
out_names = ['MRI_in_PSOCT']

for sl in ori_dir.glob('*.nii.gz'):
    out_names.append(sl.name.replace('.nii.gz',''))

subprocess.Popen(["python",
                "../visualize_zarr_ng/visualize_zarr.py",
                *inp_zarr,
                "--port", str(8050),
                "--name", *out_names])

<Popen: returncode: None args: ['python', '../visualize_zarr_ng/visualize_za...>

### Notebook for converting 2D slices to a 3D Zarr for direct visualisation

In [13]:
from pathlib import Path
import zarr
from fsl.data.image import Image
import numpy as np
# from numcodecs import Blosc

out_folder = Path('/Users/Vasilis/Downloads/CMC_results/Zarr_test')
# slice_dir = Path('/Users/Vasilis/Downloads/CMC_results/Zarr_test/sliced')
slice_dir = Path('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409/Retardance')
slice_files = sorted(slice_dir.glob('*.nii.gz'))

# Create output OME-Zarr group (overwrite each run)
out_folder.mkdir(parents=True, exist_ok=True)
out_name = out_folder / 'vol_3D_multiscales.zarr'
root = zarr.open_group(out_name, mode='w', zarr_format=3)

# Input dimensions
t0, y0, x0 = Image(slice_files[0]).shape
z = len(slice_files)
y = y0
x = x0

# Compression/chunk settings optimized for large data
full_res_chunk = (1, min(1024, y), min(1024, x))
# full_res_codec = [Blosc(cname='zstd', clevel=3, shuffle=Blosc.BITSHUFFLE)]
# pyramid_codec = [Blosc(cname='zstd', clevel=5, shuffle=Blosc.BITSHUFFLE)]

# Full-resolution array
arr0 = root.create_array(
    '0',
    shape=(z, y, x),
    chunks=full_res_chunk,
    dtype='float32',
    # compressors=full_res_codec,
)

# Adaptive multiscale schedule:
# - Start by downsampling only Y/X (fz=1)
# - When Y/X become coarse enough, include Z downsampling (fz=2)
max_levels = 6
xy_switch_threshold = 512   # start Z downsampling once min(Y, X) <= this
min_xy_size = 32            # stop once XY become too small

levels = []
cum_fz, cum_fy, cum_fx = 1, 1, 1
cur_z, cur_y, cur_x = z, y, x

for level_idx in range(1, max_levels + 1):
    if min(cur_y, cur_x) < min_xy_size:
        break

    use_z_downsample = (min(cur_y, cur_x) <= xy_switch_threshold) and (cur_z >= 2)
    step_fz = 2 if use_z_downsample else 1
    step_fy = 2 if cur_y >= 2 else 1
    step_fx = 2 if cur_x >= 2 else 1

    if (step_fz, step_fy, step_fx) == (1, 1, 1):
        break

    cum_fz *= step_fz
    cum_fy *= step_fy
    cum_fx *= step_fx

    out_z = z // cum_fz
    out_y = y // cum_fy
    out_x = x // cum_fx
    if out_z == 0 or out_y == 0 or out_x == 0:
        break

    level_name = f'scale{level_idx}'
    level_group = root.create_group(level_name)
    level_array = level_group.create_array(
        '0',
        shape=(out_z, out_y, out_x),
        chunks=(1, min(512, out_y), min(512, out_x)),
        dtype='float32', # consider changing to float16
        # compressors=pyramid_codec,
    )

    levels.append({
        'name': level_name,
        'array': level_array,
        'cum_fz': cum_fz,
        'cum_fy': cum_fy,
        'cum_fx': cum_fx,
        'out_z': out_z,
        'out_y': out_y,
        'out_x': out_x,
        'z_trim': out_z * cum_fz,
        'y_trim': out_y * cum_fy,
        'x_trim': out_x * cum_fx,
        'buffer': np.zeros((out_y, out_x), dtype=np.float32),
        'buffer_count': 0,
        'z_out_idx': 0,
    })

    cur_z, cur_y, cur_x = out_z, out_y, out_x

# Stream write: one input slice in memory at a time
overall_min = float('inf')
overall_max = float('-inf')

for i, p in enumerate(slice_files):
    data = np.asarray(Image(p).data, dtype=np.float32).squeeze()
    if data.shape != (y, x):
        raise ValueError(f'shape mismatch for {p.name}: {data.shape} != {(y, x)}')

    # Write full resolution
    arr0[i, :, :] = data

    # Incremental min/max
    smin = float(data.min())
    smax = float(data.max())
    if smin < overall_min:
        overall_min = smin
    if smax > overall_max:
        overall_max = smax

    # Write pyramid levels from same source slice
    for lvl in levels:
        if i >= lvl['z_trim']:
            continue

        d2 = data[:lvl['y_trim'], :lvl['x_trim']]
        xy_down = d2.reshape(
            lvl['out_y'], lvl['cum_fy'], lvl['out_x'], lvl['cum_fx']
        ).mean(axis=(1, 3), dtype=np.float32)

        if lvl['cum_fz'] == 1:
            lvl['array'][i, :, :] = xy_down.astype(np.float16, copy=False)
        else:
            lvl['buffer'] += xy_down
            lvl['buffer_count'] += 1
            if lvl['buffer_count'] == lvl['cum_fz']:
                lvl['array'][lvl['z_out_idx'], :, :] = (lvl['buffer'] / lvl['cum_fz']).astype(np.float16, copy=False)
                lvl['z_out_idx'] += 1
                lvl['buffer'].fill(0)
                lvl['buffer_count'] = 0

# Metadata
root.attrs['data_min'] = float(overall_min)
root.attrs['data_max'] = float(overall_max)

datasets = [
    {
        'path': '0',
        'coordinateTransformations': [
            {'type': 'scale', 'scale': [1.0, 1.0, 1.0]}
        ]
    }
]

for lvl in levels:
    datasets.append({
        'path': f"{lvl['name']}/0",
        'coordinateTransformations': [
            {
                'type': 'scale',
                'scale': [float(lvl['cum_fz']), float(lvl['cum_fy']), float(lvl['cum_fx'])]
            }
        ]
    })

root.attrs['ome'] = {
    'version': '0.5',
    'multiscales': [{
        'axes': [
            {'name': 'z', 'type': 'space'},
            {'name': 'y', 'type': 'space'},
            {'name': 'x', 'type': 'space'},
        ],
        'datasets': datasets,
        'name': 'volume',
        '@type': 'ngff:Image',
    }],
}

print('Done. Wrote scale0 +', len(levels), 'adaptive pyramid levels (XY-first, then XYZ).')
print('Min/Max:', root.attrs['data_min'], root.attrs['data_max'])
for lvl in levels:
    print(lvl['name'], 'shape=', lvl['array'].shape, 'cum_factors=', (lvl['cum_fz'], lvl['cum_fy'], lvl['cum_fx']))

Done. Wrote scale0 + 6 adaptive pyramid levels (XY-first, then XYZ).
Min/Max: 0.0 1.5698065757751465
scale1 shape= (245, 6350, 4100) cum_factors= (1, 2, 2)
scale2 shape= (245, 3175, 2050) cum_factors= (1, 4, 4)
scale3 shape= (245, 1587, 1025) cum_factors= (1, 8, 8)
scale4 shape= (245, 793, 512) cum_factors= (1, 16, 16)
scale5 shape= (122, 396, 256) cum_factors= (2, 32, 32)
scale6 shape= (61, 198, 128) cum_factors= (4, 64, 64)


In [ ]:
import subprocess
# out_names = ['MRI_in_PSOCT', 'Ori_slide_deck']
out_names = ['MRI_in_PSOCT', 'Ret_slide_deck']
inp_zarr = [str(out_folder / 'dti_FA_in_PSOCT.zarr'),
            str(out_name)]

proc = subprocess.Popen(["python",
                        "../visualize_zarr_ng/visualize_zarr.py",
                        *inp_zarr,
                        "--port", str(8066),
                        "--name", *out_names])

In [15]:
proc.terminate()

For neuroglancer view:

z in 0 to 244
y = 5291
x = 4252

https://neuroglancer-demo.appspot.com/#!%7B%22dimensions%22:%7B%22z%22:%5B1%2C%22%22%5D%2C%22y%22:%5B1%2C%22%22%5D%2C%22x%22:%5B1%2C%22%22%5D%7D%2C%22position%22:%5B120.16311645507812%2C5462.046875%2C2604.541748046875%5D%2C%22crossSectionScale%22:14.984887768412671%2C%22projectionScale%22:16384%2C%22layers%22:%5B%7B%22type%22:%22image%22%2C%22source%22:%22zarr://http://127.0.0.1:8065/dti_FA_in_PSOCT.zarr/%22%2C%22tab%22:%22rendering%22%2C%22shaderControls%22:%7B%22normalized%22:%7B%22range%22:%5B-65536%2C65536%5D%2C%22window%22:%5B-65536%2C65536%5D%7D%7D%2C%22name%22:%22MRI_in_PSOCT%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22image%22%2C%22source%22:%22zarr://http://127.0.0.1:8065/vol_3D.zarr/%22%2C%22tab%22:%22source%22%2C%22name%22:%22Ret_slide_deck%22%7D%5D%2C%22selectedLayer%22:%7B%22visible%22:true%2C%22layer%22:%22MRI_in_PSOCT%22%7D%2C%22layout%22:%224panel%22%2C%22layerListPanel%22:%7B%22visible%22:true%7D%7D